# SiPM Localization Model Comparison

This notebook compares the current localization methods on the same 1000 random single-muon test events:

- best individual-event kNN
- random forest
- gradient boosting
- projection centroid

Each row shows the radial-error histogram, headline error values, and true-position error map.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "analysis").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_FIG = PROJECT_ROOT / "analysis/model_comparison_dashboard.png"
OUTPUT_SUMMARY = PROJECT_ROOT / "analysis/model_comparison_summary.csv"

PREDICTION_FILES = {
    "kNN events": PROJECT_ROOT / "analysis/knn_sipm_best_predictions_events.csv",
    "Random forest": PROJECT_ROOT / "analysis/random_forest_sipm_predictions.csv",
    "Gradient boosting": PROJECT_ROOT / "analysis/gradient_boosting_sipm_predictions.csv",
    "Projection centroid": PROJECT_ROOT / "analysis/centroid_sipm_predictions.csv",
}

## Load Predictions

In [ ]:
def standardize_prediction_frame(name: str, path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if name == "Projection centroid":
        df = df.loc[df["dataset"] == "test"].copy()
        df["pred_x_cm"] = df["pred_x_projection_cm"]
        df["pred_z_cm"] = df["pred_z_projection_cm"]
    required = ["event_id", "true_x_cm", "true_z_cm", "pred_x_cm", "pred_z_cm"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f"{path} is missing columns: {missing}")
    out = df[required].copy()
    out["method"] = name
    out["err_x_cm"] = out["pred_x_cm"] - out["true_x_cm"]
    out["err_z_cm"] = out["pred_z_cm"] - out["true_z_cm"]
    out["err_r_cm"] = np.hypot(out["err_x_cm"], out["err_z_cm"])
    return out.sort_values("event_id").reset_index(drop=True)


predictions = pd.concat(
    [standardize_prediction_frame(name, path) for name, path in PREDICTION_FILES.items()],
    ignore_index=True,
)
predictions.groupby("method").size()

## Metrics

In [ ]:
def summarize(group: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "n_events": len(group),
        "mean_radial_err_cm": group["err_r_cm"].mean(),
        "median_radial_err_cm": group["err_r_cm"].median(),
        "p68_radial_err_cm": group["err_r_cm"].quantile(0.68),
        "p95_radial_err_cm": group["err_r_cm"].quantile(0.95),
        "mean_err_x_cm": group["err_x_cm"].mean(),
        "sigma_err_x_cm": group["err_x_cm"].std(ddof=1),
        "mean_err_z_cm": group["err_z_cm"].mean(),
        "sigma_err_z_cm": group["err_z_cm"].std(ddof=1),
    })


summary = predictions.groupby("method", sort=False).apply(summarize, include_groups=False).reset_index()
summary = summary.sort_values("p68_radial_err_cm").reset_index(drop=True)
summary.to_csv(OUTPUT_SUMMARY, index=False)
summary

## One-Page Dashboard

In [ ]:
method_order = summary["method"].tolist()
max_err = min(50, predictions["err_r_cm"].quantile(0.995))
vmax = min(35, predictions["err_r_cm"].quantile(0.98))

fig, axes = plt.subplots(
    nrows=len(method_order),
    ncols=3,
    figsize=(16, 4 * len(method_order)),
    gridspec_kw={"width_ratios": [1.25, 0.95, 1.25]},
)

for row_idx, method in enumerate(method_order):
    data = predictions.loc[predictions["method"] == method]
    metrics = summary.loc[summary["method"] == method].iloc[0]

    ax_hist = axes[row_idx, 0]
    ax_text = axes[row_idx, 1]
    ax_map = axes[row_idx, 2]

    ax_hist.hist(data["err_r_cm"], bins=np.linspace(0, max_err, 55), color="#4477AA", alpha=0.85)
    ax_hist.axvline(metrics["mean_radial_err_cm"], color="#CC3311", lw=2, label="mean")
    ax_hist.axvline(metrics["p68_radial_err_cm"], color="#228833", lw=2, label="p68")
    ax_hist.set_title(f"{method}: radial error distribution")
    ax_hist.set_xlabel("radial error [cm]")
    ax_hist.set_ylabel("events")
    ax_hist.legend(frameon=False)

    ax_text.axis("off")
    text = (
        f"{method}\n\n"
        f"Mean radial:   {metrics['mean_radial_err_cm']:.2f} cm\n"
        f"Median radial: {metrics['median_radial_err_cm']:.2f} cm\n"
        f"p68 radial:    {metrics['p68_radial_err_cm']:.2f} cm\n"
        f"p95 radial:    {metrics['p95_radial_err_cm']:.2f} cm\n\n"
        f"x mean/sigma:  {metrics['mean_err_x_cm']:.2f} / {metrics['sigma_err_x_cm']:.2f} cm\n"
        f"z mean/sigma:  {metrics['mean_err_z_cm']:.2f} / {metrics['sigma_err_z_cm']:.2f} cm"
    )
    ax_text.text(0.0, 0.95, text, va="top", ha="left", fontsize=13, family="monospace")

    sc = ax_map.scatter(
        data["true_x_cm"],
        data["true_z_cm"],
        c=data["err_r_cm"],
        s=18,
        cmap="viridis",
        vmin=0,
        vmax=vmax,
    )
    ax_map.set_title(f"{method}: error by true hit position")
    ax_map.set_xlabel("true x [cm]")
    ax_map.set_ylabel("true z [cm]")
    ax_map.set_aspect("equal", adjustable="box")
    ax_map.set_xlim(-50, 50)
    ax_map.set_ylim(-50, 50)
    cbar = fig.colorbar(sc, ax=ax_map)
    cbar.set_label("radial error [cm]")

fig.suptitle("Single-hit SiPM localization comparison", fontsize=18, y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.985])
fig.savefig(OUTPUT_FIG, dpi=180)
OUTPUT_FIG